# Explore Indicators: Adaptive Moving Averages and Regime Detection

This notebook explores ta_lab2's **Adaptive Moving Average (AMA) engine** and **regime detection system** — two core components of the v0.9.0 research layer.

AMAs are trend-following indicators that adapt their responsiveness based on market conditions. Where a traditional EMA applies a fixed smoothing coefficient, AMAs reduce lag during trending markets and increase it during choppy, sideways action.

ta_lab2's regime system labels each bar with a composite `l2_label` that combines a trend state and a volatility state (e.g. `uptrend-low_vol`). Overlaying regimes on AMA charts reveals when each indicator type performs best.

---

## Table of Contents

1. [Setup](#setup)
2. [Parameters](#parameters)
3. [DB Connection and Validation](#db-connection)
4. [Load Price Bars](#load-bars)
5. [What Are Adaptive Moving Averages?](#ama-intro)
6. [Compute All 4 AMA Types](#compute-amas)
7. [AMA Comparison Chart](#ama-chart)
8. [KAMA Efficiency Ratio Deep Dive](#kama-er)
9. [Regime Detection](#regime-detection)
10. [Regime Distribution](#regime-distribution)
11. [Regime-Colored Price Chart with AMA Overlay](#regime-overlay)
12. [AMA Statistics by Regime](#ama-stats)
13. [Multi-Timeframe Preview](#multi-tf)
14. [Summary and Next Steps](#summary)

## Prerequisites

**Required database tables:**
- `cmc_price_bars_multi_tf` — OHLCV bars at all timeframes
- `cmc_regimes` — Regime labels (l0, l1, l2) per bar
- `asset_data_coverage` — Data coverage summary (rows, days, first/last ts)

Note: AMAs are computed **in-notebook from price bars** — no pre-computed AMA table is queried.

**Required packages:**
```bash
pip install jupyterlab seaborn ipykernel
```

**Kernel registration (run once):**
```bash
python -m ipykernel install --user --name ta_lab2 --display-name "ta_lab2"
```
Then select the `ta_lab2` kernel from the Jupyter kernel menu.

In [ ]:
import sys
from pathlib import Path

_ROOT = Path.cwd().parent
if str(_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(_ROOT / "src"))

import helpers
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# pio.renderers.default = "notebook"  # uncomment if charts don't render inline

import ta_lab2
print(f"ta_lab2 loaded from: {ta_lab2.__file__}")

<a id='parameters'></a>
## Parameters

Change these to explore different assets and timeframes. Re-run all cells after changing.

In [ ]:
# --- Parameters (change these to explore different assets/timeframes) ---
ASSET_ID = 1       # BTC; try 2 for ETH
TF = "1D"          # Primary timeframe; try "1W" for weekly
START_DATE = "2021-01-01"
END_DATE = "2025-12-31"

<a id='db-connection'></a>
## DB Connection and Validation

We create a single `NullPool` engine (no connection pooling — appropriate for notebooks) and validate that the asset has at least 365 days of coverage before running any analysis.

In [ ]:
engine = helpers.get_engine()
check = helpers.validate_asset_data(engine, ASSET_ID, TF, min_days=365)
print(f"Asset {ASSET_ID} ({TF}): {check['message']}")
if not check["valid"]:
    raise RuntimeError(f"Insufficient data: {check['message']}")
print(f"  Coverage: {check['n_days']} days from {check['first_ts']} to {check['last_ts']}")

<a id='load-bars'></a>
## Load Price Bars

Price bars are loaded from `cmc_price_bars_multi_tf`. The `ts` index is always UTC-aware (Windows tz pitfall handled by helpers.py).

In [ ]:
bars = helpers.load_price_bars(engine, ASSET_ID, TF, START_DATE, END_DATE)
if bars.empty:
    raise RuntimeError(f"No price bars found for asset {ASSET_ID} / {TF} in [{START_DATE}, {END_DATE}]")
print(f"Loaded {len(bars)} bars from {bars.index.min()} to {bars.index.max()}")
bars.head()

<a id='ama-intro'></a>
## What Are Adaptive Moving Averages?

A **traditional EMA** applies a fixed smoothing coefficient `alpha = 2/(period+1)` regardless of whether the market is trending or choppy. This means it lags on fast moves and overreacts to noise during consolidations.

**Adaptive Moving Averages** solve this by dynamically adjusting their smoothing coefficient based on observed market conditions:

| AMA | Core Idea | Adapts via |
|-----|-----------|------------|
| **KAMA** | Kaufman Adaptive MA — fastest/slowest SC blend | Efficiency Ratio (ER) |
| **DEMA** | Double EMA — cancels EMA lag by doubling and subtracting EMA-of-EMA | Fixed period, no adaptation |
| **TEMA** | Triple EMA — extends DEMA concept for even lower lag | Fixed period, no adaptation |
| **HMA** | Hull MA — weighted MA trick to nearly eliminate lag | Fixed period, uses WMA |

**KAMA in detail:**
- The **Efficiency Ratio (ER)** = `|net price change| / sum(|bar changes|)` over a lookback window.
- ER ≈ 1.0 in strongly trending markets; ER ≈ 0.0 in perfectly choppy markets.
- KAMA's smoothing constant blends between a fast SC (high ER) and a slow SC (low ER).

DEMA and TEMA are not truly "adaptive" — they reduce lag through formula construction rather than market sensing — but they are included in ta_lab2's AMA family as close relatives.

<a id='compute-amas'></a>
## Compute All 4 AMA Types

AMAs are computed on-the-fly from the raw close series using `compute_ama()`. No pre-computed AMA table is needed.

In [ ]:
from ta_lab2.features.ama.ama_computations import compute_ama

close = bars["close"]

kama, er = compute_ama(close, "KAMA", {"er_period": 10, "fast_period": 2, "slow_period": 30})
dema, _  = compute_ama(close, "DEMA", {"period": 21})
tema, _  = compute_ama(close, "TEMA", {"period": 21})
hma,  _  = compute_ama(close, "HMA",  {"period": 21})

ama_df = pd.DataFrame(
    {
        "close": close,
        "KAMA(10,2,30)": kama,
        "DEMA(21)": dema,
        "TEMA(21)": tema,
        "HMA(21)": hma,
        "KAMA_ER": er,
    }
)
print(f"AMA values computed: {len(ama_df)} rows, {ama_df.notna().sum().to_dict()}")
ama_df.head(30).tail()

<a id='ama-chart'></a>
## AMA Comparison Chart

All four AMAs overlaid on the raw close price. Notice:
- **TEMA and HMA** hug price most closely — lowest lag.
- **KAMA** flattens during low-ER (choppy) periods and tracks sharply during high-ER (trending) periods.
- **DEMA** sits between KAMA and TEMA in responsiveness.

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=ama_df.index,
        y=ama_df["close"],
        name="Close",
        line=dict(color="black", width=1),
        opacity=0.5,
    )
)

ama_colors = {
    "KAMA(10,2,30)": "royalblue",
    "DEMA(21)": "darkorange",
    "TEMA(21)": "green",
    "HMA(21)": "red",
}
for col, color in ama_colors.items():
    fig.add_trace(
        go.Scatter(
            x=ama_df.index,
            y=ama_df[col],
            name=col,
            line=dict(color=color, width=1.5),
        )
    )

fig.update_layout(
    title=f"AMA Comparison — Asset {ASSET_ID} ({TF})",
    xaxis_title="Date",
    yaxis_title="Price",
    height=500,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    template="plotly_white",
)
fig.show()

<a id='kama-er'></a>
## KAMA Efficiency Ratio Deep Dive

The **Efficiency Ratio (ER)** is the key insight behind KAMA. It measures directional efficiency: how much of total movement is in a single net direction.

- **ER → 1.0**: Price moves steadily in one direction. KAMA tracks closely.
- **ER → 0.0**: Price oscillates with no net direction. KAMA flattens out.

Threshold interpretation:
- **ER > 0.7**: Strong trend — KAMA uses fast smoothing
- **ER < 0.3**: Sideways / choppy — KAMA uses slow smoothing

In [ ]:
from plotly.subplots import make_subplots

fig_er = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    row_heights=[0.65, 0.35],
    subplot_titles=[f"Close vs KAMA — Asset {ASSET_ID} ({TF})", "KAMA Efficiency Ratio (ER)"],
)

# Row 1: Close + KAMA
fig_er.add_trace(
    go.Scatter(x=ama_df.index, y=ama_df["close"], name="Close", line=dict(color="black", width=1), opacity=0.5),
    row=1, col=1,
)
fig_er.add_trace(
    go.Scatter(x=ama_df.index, y=ama_df["KAMA(10,2,30)"], name="KAMA(10,2,30)", line=dict(color="royalblue", width=1.8)),
    row=1, col=1,
)

# Row 2: ER as filled area
er_valid = ama_df["KAMA_ER"].dropna()
if not er_valid.empty:
    fig_er.add_trace(
        go.Scatter(
            x=er_valid.index,
            y=er_valid,
            name="ER",
            fill="tozeroy",
            line=dict(color="steelblue", width=1),
            fillcolor="rgba(70, 130, 180, 0.3)",
        ),
        row=2, col=1,
    )
    # Threshold lines
    for threshold, color, label in [(0.7, "green", "Strong trend"), (0.3, "red", "Choppy")]:
        fig_er.add_hline(y=threshold, line_dash="dash", line_color=color, annotation_text=label, row=2, col=1)

fig_er.update_layout(height=600, template="plotly_white", showlegend=True)
fig_er.show()

**Reading the ER chart:**

- When ER spikes above the green (0.7) line, KAMA switches to fast-tracking mode — price moves are directional and KAMA should follow closely.
- When ER drops below the red (0.3) line, KAMA becomes nearly flat — noise dominates and tracking closely would cause excessive whipsawing.
- The blue shaded area shows the ER magnitude over time. Compare peaks/troughs to the price chart above: ER peaks typically coincide with strong trending moves.

<a id='regime-detection'></a>
## Regime Detection

ta_lab2's regime system assigns a composite label to each bar at three layers:

- **L0**: Market-level regime (based on broad market proxy — typically BTC)
- **L1**: Asset-level trend classification
- **L2**: Combined trend + volatility label (e.g. `uptrend-low_vol`, `downtrend-high_vol`)

L2 is the primary label used for analysis. It combines a **trend state** (`uptrend`/`sideways`/`downtrend`) with a **volatility state** (`low_vol`/`high_vol`/`extreme_vol`) to give a nuanced market description.

Regimes use a **HysteresisTracker** with 3-bar hold to prevent whipsawing between labels — a transition is only accepted if the new regime persists.

In [ ]:
regimes = helpers.load_regimes(engine, ASSET_ID, TF, START_DATE, END_DATE)

if regimes.empty:
    print("WARNING: No regime data found. Regime sections will be skipped.")
    print("Run: python -m ta_lab2.scripts.regimes.refresh_cmc_regimes --all")
    HAS_REGIMES = False
else:
    HAS_REGIMES = True
    print(f"Loaded {len(regimes)} regime bars")
    print(f"L2 labels present: {regimes['l2_label'].nunique()} unique values")
    print(regimes[[col for col in ['l0_label', 'l1_label', 'l2_label'] if col in regimes.columns]].head())

<a id='regime-distribution'></a>
## Regime Distribution

How are regime labels distributed over our date range? Ideally we see a balanced mix — heavy skew toward one regime might indicate the analysis window is too short.

In [ ]:
if HAS_REGIMES and "l2_label" in regimes.columns:
    counts = regimes["l2_label"].value_counts().sort_values(ascending=False)
    pct = (counts / counts.sum() * 100).round(1)

    fig_dist, ax = plt.subplots(figsize=(10, 4))
    bars_plot = ax.bar(counts.index, counts.values, color="steelblue", alpha=0.8, edgecolor="white")
    ax.set_xlabel("L2 Regime Label")
    ax.set_ylabel("Number of Bars")
    ax.set_title(f"Regime Distribution — Asset {ASSET_ID} ({TF}), {START_DATE} to {END_DATE}")
    ax.tick_params(axis="x", rotation=45)

    for bar, (label, p) in zip(bars_plot, pct.items()):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.5,
            f"{p}%",
            ha="center",
            va="bottom",
            fontsize=9,
        )

    plt.tight_layout()
    plt.show()

    print("\nRegime counts:")
    print(pd.DataFrame({"count": counts, "pct": pct}))
else:
    print("Skipping regime distribution — no regime data available.")

<a id='regime-overlay'></a>
## Regime-Colored Price Chart with AMA Overlay

The most informative view: price bars colored by their L2 regime label, with KAMA overlaid.

Each regime is rendered as a **vertical rectangle (vrect)** spanning consecutive bars with the same label. This approach groups consecutive same-label bars into single shapes, avoiding the performance penalty of one vrect per bar.

In [ ]:
# Regime color palette — extend as needed for new labels
REGIME_COLORS = {
    "uptrend-low_vol":      "rgba(0, 200, 100, 0.15)",
    "uptrend-high_vol":     "rgba(0, 200, 100, 0.30)",
    "uptrend-extreme_vol":  "rgba(0, 200, 100, 0.45)",
    "sideways-low_vol":     "rgba(180, 180, 180, 0.15)",
    "sideways-high_vol":    "rgba(180, 180, 180, 0.30)",
    "sideways-extreme_vol": "rgba(180, 180, 180, 0.45)",
    "downtrend-low_vol":    "rgba(220, 50, 50, 0.15)",
    "downtrend-high_vol":   "rgba(220, 50, 50, 0.30)",
    "downtrend-extreme_vol":"rgba(220, 50, 50, 0.45)",
}
DEFAULT_REGIME_COLOR = "rgba(200, 200, 200, 0.15)"


def build_regime_vrects(regimes_df: pd.DataFrame, label_col: str = "l2_label") -> list[dict]:
    """Group consecutive same-label bars into vrect blocks."""
    if regimes_df.empty or label_col not in regimes_df.columns:
        return []
    blocks = []
    current_label = None
    block_start = None
    prev_ts = None

    for ts, row in regimes_df.iterrows():
        label = row[label_col]
        if label != current_label:
            if current_label is not None:
                blocks.append({"x0": block_start, "x1": prev_ts, "label": current_label})
            current_label = label
            block_start = ts
        prev_ts = ts

    if current_label is not None:
        blocks.append({"x0": block_start, "x1": prev_ts, "label": current_label})
    return blocks


if HAS_REGIMES:
    fig_overlay = go.Figure()

    # Add vrect shading grouped by consecutive blocks
    blocks = build_regime_vrects(regimes, "l2_label")
    for block in blocks:
        color = REGIME_COLORS.get(block["label"], DEFAULT_REGIME_COLOR)
        fig_overlay.add_vrect(
            x0=block["x0"],
            x1=block["x1"],
            fillcolor=color,
            opacity=1,
            line_width=0,
            layer="below",
        )

    # Close price
    fig_overlay.add_trace(
        go.Scatter(
            x=bars.index,
            y=bars["close"],
            name="Close",
            line=dict(color="black", width=1),
            opacity=0.6,
        )
    )

    # KAMA overlay
    fig_overlay.add_trace(
        go.Scatter(
            x=ama_df.index,
            y=ama_df["KAMA(10,2,30)"],
            name="KAMA(10,2,30)",
            line=dict(color="royalblue", width=2),
        )
    )

    fig_overlay.update_layout(
        title=f"Regime-Colored Price Chart + KAMA — Asset {ASSET_ID} ({TF})",
        xaxis_title="Date",
        yaxis_title="Price",
        height=550,
        template="plotly_white",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    )
    fig_overlay.show()
else:
    # Fallback: just plot close + KAMA without regime coloring
    fig_no_regime = go.Figure()
    fig_no_regime.add_trace(go.Scatter(x=bars.index, y=bars["close"], name="Close", line=dict(color="black", width=1)))
    fig_no_regime.add_trace(go.Scatter(x=ama_df.index, y=ama_df["KAMA(10,2,30)"], name="KAMA(10,2,30)", line=dict(color="royalblue", width=2)))
    fig_no_regime.update_layout(title=f"Close + KAMA (no regime data) — Asset {ASSET_ID} ({TF})", height=450, template="plotly_white")
    fig_no_regime.show()
    print("Tip: Run the regime refresh to add regime coloring.")

**Reading the regime overlay:**

- **Green regions** — uptrend regime. KAMA should be tracking price closely (high ER).
- **Gray regions** — sideways/choppy regime. Expect KAMA to flatten, DEMA/HMA to oscillate.
- **Red regions** — downtrend regime. AMAs should trail below price.

Darker shading within a color group indicates higher volatility (e.g. `uptrend-high_vol` is darker green than `uptrend-low_vol`).

<a id='ama-stats'></a>
## AMA Statistics by Regime

How do AMA deviations from close behave across regimes? The deviation `(ama - close) / close` shows how much each AMA leads or lags. In trending markets, we expect the lagging AMA to be below close (uptrend) or above close (downtrend).

In [ ]:
if HAS_REGIMES:
    # Align regimes to AMA dataframe index
    regime_aligned = regimes[["l2_label"]].reindex(ama_df.index)

    # Compute relative deviations: (ama - close) / close
    dev_df = pd.DataFrame(index=ama_df.index)
    for col in ["KAMA(10,2,30)", "DEMA(21)", "TEMA(21)", "HMA(21)"]:
        dev_df[col] = (ama_df[col] - ama_df["close"]) / ama_df["close"]
    dev_df["l2_label"] = regime_aligned["l2_label"]

    stats = (
        dev_df.dropna(subset=["l2_label"])
        .groupby("l2_label")[["KAMA(10,2,30)", "DEMA(21)", "TEMA(21)", "HMA(21)"]]
        .agg(["mean", "std", "count"])
        .round(4)
    )
    print("AMA deviation from close (ama - close) / close grouped by L2 regime:")
    print("(Negative = AMA below close; Positive = AMA above close)")
    display(stats)  # noqa: F821 — Jupyter built-in
else:
    # Compute overall stats without regime breakdown
    dev_df = pd.DataFrame(index=ama_df.index)
    for col in ["KAMA(10,2,30)", "DEMA(21)", "TEMA(21)", "HMA(21)"]:
        dev_df[col] = (ama_df[col] - ama_df["close"]) / ama_df["close"]

    print("AMA deviation from close (overall, no regime breakdown):")
    print(dev_df.describe().round(4))

<a id='multi-tf'></a>
## Multi-Timeframe Preview

AMAs are available at any timeframe in `cmc_price_bars_multi_tf`. Here we quickly compute HMA(21) on weekly bars as a preview of the multi-TF workflow.

Weekly bars naturally have fewer bars per year, so we need at least 2 years of daily data (~104 weekly bars) to get meaningful results.

In [ ]:
WEEKLY_TF = "1W"

check_w = helpers.validate_asset_data(engine, ASSET_ID, WEEKLY_TF, min_days=180)
print(f"Weekly data check: {check_w['message']}")

if check_w["valid"]:
    bars_w = helpers.load_price_bars(engine, ASSET_ID, WEEKLY_TF, START_DATE, END_DATE)
    if len(bars_w) >= 25:  # Need at least 25 bars for HMA(21) warmup
        hma_w, _ = compute_ama(bars_w["close"], "HMA", {"period": 21})

        fig_w = go.Figure()
        fig_w.add_trace(
            go.Scatter(x=bars_w.index, y=bars_w["close"], name="Close (1W)", line=dict(color="black", width=1), opacity=0.5)
        )
        fig_w.add_trace(
            go.Scatter(x=bars_w.index, y=hma_w, name="HMA(21) on 1W", line=dict(color="purple", width=2))
        )
        fig_w.update_layout(
            title=f"Weekly HMA(21) — Asset {ASSET_ID} ({WEEKLY_TF})",
            xaxis_title="Date",
            yaxis_title="Price",
            height=400,
            template="plotly_white",
        )
        fig_w.show()
        print(f"Weekly bars: {len(bars_w)}, valid HMA values: {hma_w.notna().sum()}")
    else:
        print(f"Only {len(bars_w)} weekly bars available — need at least 25 for HMA(21). Skipping chart.")
else:
    print(f"Skipping weekly multi-TF preview — insufficient data for {ASSET_ID}/{WEEKLY_TF}.")
    print("Tip: Run the multi-TF bar refresh to populate weekly bars.")

<a id='summary'></a>
## Summary and Next Steps

### What We Covered

1. **AMA Computation** — On-the-fly computation of KAMA, DEMA, TEMA, HMA from raw price bars using `compute_ama()`. No pre-computed AMA table required.

2. **KAMA Efficiency Ratio** — The ER is the key signal inside KAMA. High ER = trend, low ER = chop. The ER can itself be used as a signal for regime classification.

3. **Regime Detection** — ta_lab2's HysteresisTracker assigns stable L2 labels combining trend + volatility state. Regime coloring reveals where each AMA type performs best.

4. **Regime-Based Analysis** — AMA deviation from close, segmented by regime, shows how lag characteristics differ across market environments.

5. **Multi-TF Preview** — The same computation pipeline works at any timeframe available in `cmc_price_bars_multi_tf`.

### Next Steps

- **Notebook 02 (IC Analysis)** — Measure predictive power of each AMA's deviation-from-close as a feature. Use IC (Information Coefficient) with BH correction to identify statistically robust signals.

- **Notebook 03 (Backtest)** — Run backtests conditioned on regime labels. Compare AMA crossover strategies in trending vs sideways regimes.

- **Promote to feature registry** — If AMA-ER shows IC > 0.05 at 5-day horizon (p < 0.05 after BH correction), promote to `dim_feature_registry` and include in `cmc_feature_experiments`.

### Useful Commands

```bash
# Refresh price bars (needed for on-the-fly AMA)
python -m ta_lab2.scripts.run_daily_refresh --bars --all

# Refresh regime labels
python -m ta_lab2.scripts.run_daily_refresh --regimes --all

# Run full IC evaluation
python -m ta_lab2.scripts.ic.run_ic_eval --asset-id 1 --tf 1D --train-start 2021-01-01 --train-end 2024-12-31
```